In [0]:
# ============================================================
# PROJECT: Banking Transactions Data Pipeline
# FILE NAME: banking_transactions_spark_pipeline.ipynb
# TECHNOLOGY: Apache Spark + Databricks
# ARCHITECTURE: Bronze – Silver – Gold
#
# DESCRIPTION:
# This notebook implements an end-to-end Spark pipeline to
# ingest, clean, validate, transform, and analyze banking
# transaction data with data quality checks and Delta Lake.
# ============================================================


In [0]:
# ============================================================
# 1. DATA INGESTION
# PURPOSE:
# - Load raw banking transaction data from CSV into Spark
# - Infer schema and inspect structure
# ============================================================


# Read CSV file
df = spark.read.option("header", True)\
    .option("inferSchema", True)\
    .csv("/Volumes/workspace/pyspark/spark/csv/transactions.csv")

# Show data
df.show()


+--------------+-----------+----------------+------+------------+-------+---------+
|transaction_id|customer_id|transaction_date|amount|payment_mode| status|     city|
+--------------+-----------+----------------+------+------------+-------+---------+
|          T001|       C101|      2025-01-01|   500|         UPI|SUCCESS|    Delhi|
|          T002|       C102|      2025-01-01|  1500|        Card|SUCCESS|   Mumbai|
|          T003|       C103|      2025-01-02|  NULL|         UPI| FAILED|Bangalore|
|          T004|       C101|      2025-01-02|   700|  NetBanking|SUCCESS|    Delhi|
|          T005|       C104|      2025-01-03|  -200|        Card|SUCCESS|     Pune|
|          T006|       C105|      2025-01-03|  1200|        NULL|SUCCESS|  Chennai|
|          T006|       C105|      2025-01-03|  1200|        NULL|SUCCESS|  Chennai|
|          T007|       C106|      2025-01-04|   900|         UPI|   NULL|Hyderabad|
+--------------+-----------+----------------+------+------------+-------+---

In [0]:
df.printSchema() # print schema

root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- amount: integer (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- status: string (nullable = true)
 |-- city: string (nullable = true)



In [0]:
# ============================================================
# 2. DATA QUALITY CHECKS
# PURPOSE:
# - Identify missing (null) values
# - Understand data issues before cleaning
# ============================================================


from pyspark.sql.functions import col, count, when

null_counts = df.select(
    [count(when(col(c).isNull(), c)).alias(c) for c in df.columns]
)

null_counts.show()


+--------------+-----------+----------------+------+------------+------+----+
|transaction_id|customer_id|transaction_date|amount|payment_mode|status|city|
+--------------+-----------+----------------+------+------------+------+----+
|             0|          0|               0|     1|           2|     1|   0|
+--------------+-----------+----------------+------+------------+------+----+



In [0]:
# ============================================================
# 3. DATA CLEANING & BUSINESS RULE VALIDATION
# PURPOSE:
# - Handle missing values
# - Remove duplicate transactions
# - Fix invalid (negative) transaction amounts
# ============================================================


df_filled=df.fillna({
    'amount': 0,
    'payment_mode':'Unknown',
    'status':'Unknown'
})

df_filled.show()

+--------------+-----------+----------------+------+------------+-------+---------+
|transaction_id|customer_id|transaction_date|amount|payment_mode| status|     city|
+--------------+-----------+----------------+------+------------+-------+---------+
|          T001|       C101|      2025-01-01|   500|         UPI|SUCCESS|    Delhi|
|          T002|       C102|      2025-01-01|  1500|        Card|SUCCESS|   Mumbai|
|          T003|       C103|      2025-01-02|     0|         UPI| FAILED|Bangalore|
|          T004|       C101|      2025-01-02|   700|  NetBanking|SUCCESS|    Delhi|
|          T005|       C104|      2025-01-03|  -200|        Card|SUCCESS|     Pune|
|          T006|       C105|      2025-01-03|  1200|     Unknown|SUCCESS|  Chennai|
|          T006|       C105|      2025-01-03|  1200|     Unknown|SUCCESS|  Chennai|
|          T007|       C106|      2025-01-04|   900|         UPI|Unknown|Hyderabad|
+--------------+-----------+----------------+------+------------+-------+---

In [0]:
df_no_duplicate=df_filled.dropDuplicates(["transaction_id"])

df_no_duplicate.show()

+--------------+-----------+----------------+------+------------+-------+---------+
|transaction_id|customer_id|transaction_date|amount|payment_mode| status|     city|
+--------------+-----------+----------------+------+------------+-------+---------+
|          T007|       C106|      2025-01-04|   900|         UPI|Unknown|Hyderabad|
|          T001|       C101|      2025-01-01|   500|         UPI|SUCCESS|    Delhi|
|          T006|       C105|      2025-01-03|  1200|     Unknown|SUCCESS|  Chennai|
|          T005|       C104|      2025-01-03|  -200|        Card|SUCCESS|     Pune|
|          T003|       C103|      2025-01-02|     0|         UPI| FAILED|Bangalore|
|          T004|       C101|      2025-01-02|   700|  NetBanking|SUCCESS|    Delhi|
|          T002|       C102|      2025-01-01|  1500|        Card|SUCCESS|   Mumbai|
+--------------+-----------+----------------+------+------------+-------+---------+



In [0]:
# Fix negative transaction amounts

from pyspark.sql.functions import when

df_clean = df_no_duplicate.withColumn(
    "amount",
    when(df_no_duplicate.amount < 0, 0).otherwise(df_no_duplicate.amount)
)

df_clean.show()


+--------------+-----------+----------------+------+------------+-------+---------+
|transaction_id|customer_id|transaction_date|amount|payment_mode| status|     city|
+--------------+-----------+----------------+------+------------+-------+---------+
|          T007|       C106|      2025-01-04|   900|         UPI|Unknown|Hyderabad|
|          T001|       C101|      2025-01-01|   500|         UPI|SUCCESS|    Delhi|
|          T006|       C105|      2025-01-03|  1200|     Unknown|SUCCESS|  Chennai|
|          T005|       C104|      2025-01-03|     0|        Card|SUCCESS|     Pune|
|          T003|       C103|      2025-01-02|     0|         UPI| FAILED|Bangalore|
|          T004|       C101|      2025-01-02|   700|  NetBanking|SUCCESS|    Delhi|
|          T002|       C102|      2025-01-01|  1500|        Card|SUCCESS|   Mumbai|
+--------------+-----------+----------------+------+------------+-------+---------+



In [0]:
# Total transaction amount by payment mode

df_clean.groupBy("payment_mode") \
    .sum("amount") \
    .withColumnRenamed("sum(amount)", "total_amount") \
    .show()


+------------+------------+
|payment_mode|total_amount|
+------------+------------+
|         UPI|        1400|
|     Unknown|        1200|
|        Card|        1500|
|  NetBanking|         700|
+------------+------------+



In [0]:
# Save cleaned data to the correct path

df_clean.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("/Volumes/workspace/pyspark/spark/csv/transactions_cleaned")


In [0]:
# ============================================================
# 4. BRONZE LAYER (RAW DATA)
# PURPOSE:
# - Store raw, unmodified data
# - Enable audit, traceability, and reprocessing
# ============================================================


df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("/Volumes/workspace/pyspark/spark/csv/bronze_transactions")


In [0]:
# ============================================================
# 5. SILVER LAYER (CLEANED DATA)
# PURPOSE:
# - Store cleaned and validated data
# - Remove data quality issues
# - Prepare data for analytics
# ============================================================


df_clean.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("/Volumes/workspace/pyspark/spark/csv/silver_transactions")


In [0]:
# ============================================================
# 6. GOLD LAYER (ANALYTICS READY DATA)
# PURPOSE:
# - Generate aggregated business metrics
# - Provide analytics-ready datasets for reporting
# ============================================================

gold_df = df_clean.groupBy("city") \
    .sum("amount") \
    .withColumnRenamed("sum(amount)", "total_transaction_amount")

gold_df.show()


+---------+------------------------+
|     city|total_transaction_amount|
+---------+------------------------+
|Hyderabad|                     900|
|    Delhi|                    1200|
|  Chennai|                    1200|
|     Pune|                       0|
|Bangalore|                       0|
|   Mumbai|                    1500|
+---------+------------------------+



In [0]:
gold_df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("/Volumes/workspace/pyspark/spark/csv/gold_transactions_by_city")


In [0]:
# Upgrade Step 4: Save Silver layer in Delta format

df_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/pyspark/spark/delta/silver_transactions")


In [0]:
# Read Silver data from Delta format

silver_delta_df = spark.read.format("delta") \
    .load("/Volumes/workspace/pyspark/spark/delta/silver_transactions")

silver_delta_df.show()


+--------------+-----------+----------------+------+------------+-------+---------+
|transaction_id|customer_id|transaction_date|amount|payment_mode| status|     city|
+--------------+-----------+----------------+------+------------+-------+---------+
|          T007|       C106|      2025-01-04|   900|         UPI|Unknown|Hyderabad|
|          T001|       C101|      2025-01-01|   500|         UPI|SUCCESS|    Delhi|
|          T006|       C105|      2025-01-03|  1200|     Unknown|SUCCESS|  Chennai|
|          T005|       C104|      2025-01-03|     0|        Card|SUCCESS|     Pune|
|          T003|       C103|      2025-01-02|     0|         UPI| FAILED|Bangalore|
|          T004|       C101|      2025-01-02|   700|  NetBanking|SUCCESS|    Delhi|
|          T002|       C102|      2025-01-01|  1500|        Card|SUCCESS|   Mumbai|
+--------------+-----------+----------------+------+------------+-------+---------+



In [0]:
# Create temporary SQL view

silver_delta_df.createOrReplaceTempView("transactions_silver")


In [0]:
spark.sql("""
SELECT
    city,
    payment_mode,
    SUM(amount) AS total_amount
FROM transactions_silver
GROUP BY city, payment_mode
ORDER BY total_amount DESC
""").show()


+---------+------------+------------+
|     city|payment_mode|total_amount|
+---------+------------+------------+
|   Mumbai|        Card|        1500|
|  Chennai|     Unknown|        1200|
|Hyderabad|         UPI|         900|
|    Delhi|  NetBanking|         700|
|    Delhi|         UPI|         500|
|Bangalore|         UPI|           0|
|     Pune|        Card|           0|
+---------+------------+------------+



In [0]:
# Save Gold layer in Delta format

gold_delta_df = spark.sql("""
SELECT
    city,
    payment_mode,
    SUM(amount) AS total_amount
FROM transactions_silver
GROUP BY city, payment_mode
""")

gold_delta_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/pyspark/spark/delta/gold_transactions")


In [0]:
# ============================================================
# 7. DATA QUALITY SUMMARY & GOVERNANCE
# PURPOSE:
# - Capture data health metrics
# - Support monitoring, controls, and governance
# ============================================================
# Total records
total_records = df.count()

# Null amount count
null_amount = df.filter(col("amount").isNull()).count()

# Duplicate transaction IDs
duplicate_count = df.groupBy("transaction_id") \
    .count() \
    .filter(col("count") > 1) \
    .count()

# Negative amount count
negative_amount = df.filter(col("amount") < 0).count()


In [0]:



dq_summary = spark.createDataFrame([
    ("Total Records", total_records),
    ("Null Amount Records", null_amount),
    ("Duplicate Transactions", duplicate_count),
    ("Negative Amount Records", negative_amount)
], ["Metric", "Value"])

dq_summary.show()


+--------------------+-----+
|              Metric|Value|
+--------------------+-----+
|       Total Records|    8|
| Null Amount Records|    1|
|Duplicate Transac...|    1|
|Negative Amount R...|    1|
+--------------------+-----+



In [0]:
dq_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/pyspark/spark/delta/data_quality_summary")


In [0]:
# ============================================================
# END OF PIPELINE
# OUTPUTS:
# - Bronze Data (Raw)
# - Silver Data (Cleaned, Delta)
# - Gold Data (Aggregated, Delta)
# - Data Quality Summary
# ============================================================
